### Imports e Pré-Processamento de Dados

In [ ]:
# Imports dos dados
import numpy as np
import pandas as pd
from datetime import datetime
from feature_eng import split_data, input_h2h_features

# Imports do Modelo
import joblib
import optuna
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import root_mean_squared_error

In [ ]:
def goal_handicap(handicap) -> float | None:

    """
    Gets the handicap from the API and solves type errors,
    Also, converts it to a float and returns the current handicap if successful.
    """

    try: 
        if isinstance(handicap, str):
            if ',' in handicap:
                handicap_vals = [float(h.strip()) for h in handicap.split(',')]
                handicap = sum(handicap_vals) / len(handicap_vals)
            
            else:
                handicap = float(handicap.strip())
        
        return handicap
    
    except ValueError as ve:
        print(f"Error converting handicap '{handicap}': {ve}")

In [ ]:
df_raw = pd.read_csv('../cleaned_data/df_odds.csv', parse_dates=['date'], low_memory=False)

cols_to_save = [
    "event_id","league","date",
    "away_team","away_player","away_score",
    "home_team","home_player","home_score",
    "total_score"
    ]

# Tratamento de Alguns dados
df_raw = df_raw.sort_values("date").reset_index(drop=True)
df_raw = df_raw[df_raw['league'] == 22614]

# Inserir no dataset probabilidades implícitas
df_raw['1_3_handicap'] = df_raw['1_3_handicap'].apply(goal_handicap)


In [ ]:
train_raw, test_raw = split_data(
    df=df_raw,
    split_date=datetime(2025,8,1),
    date_column='date'
)

train_with_avg, test_with_avg, scaler = input_h2h_features(
    train_raw,
    test_raw,
    short_window=5,
    long_window=50,
    flag_threshold=1.0,
    return_scaler=True,
    drop_h2h=3
)

# manter apenas linhas com pelo menos uma flag = 1
# train_with_avg = train_with_avg[train_with_avg[flag_cols].any(axis=1)].copy()
# test_with_avg  = test_with_avg[test_with_avg[flag_cols].any(axis=1)].copy()


### Definições Iniciais para o modelo

In [ ]:
features = [
    "avg_h2h_long",
    "median_h2h_long",
    "avg_h2h_short",
    "flag_below_long",
    "flag_above_long",
    "flag_below_median",
    "flag_above_median",
    "flag_same_result",
]

### Treinamento do Modelo

In [ ]:
X_train = train_with_avg[features]
y_train = train_with_avg['total_score']

X_test = test_with_avg[features]
y_test = test_with_avg['total_score']

X = X_train.values
y = y_train.values

In [ ]:
def timeseries_cv_rmse(X, y, params, n_splits=5):
    rmse_scores = []
    tscv = TimeSeriesSplit(n_splits=n_splits)

    for train_index, val_index in tscv.split(X):
        X_tr, X_val = X[train_index], X[val_index]
        y_tr, y_val = y[train_index], y[val_index]

        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval = xgb.DMatrix(X_val, label=y_val)

        train_params = {
            "learning_rate": params["learning_rate"],
            "max_depth": params["max_depth"],
            "subsample": params["subsample"],
            "colsample_bytree": params["colsample_bytree"],
            "min_child_weight": params["min_child_weight"],
            "gamma": params["gamma"],
            "reg_alpha": params["reg_alpha"],
            "reg_lambda": params["reg_lambda"],
            "objective": "reg:squarederror",
            "eval_metric": "rmse",
            "tree_method": "hist",
            "seed": 42,
        }

        evals = [(dval, "eval"), (dtrain, "train")]

        bst = xgb.train(
            params=train_params,
            dtrain=dtrain,
            num_boost_round=params["n_estimators"],
            evals=evals,
            early_stopping_rounds=30,
            verbose_eval=False
        )

        y_pred = bst.predict(dval, iteration_range=(0, bst.best_iteration+1))
        rmse = root_mean_squared_error(y_val, y_pred)
        rmse_scores.append(rmse)

    return np.mean(rmse_scores)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 900),
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 1, 12),
        "subsample": trial.suggest_float("subsample", 0.1, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.1, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 1),
        "reg_lambda": trial.suggest_float("reg_lambda", 1, 10),
    }

    return timeseries_cv_rmse(X, y, params, n_splits=5)

In [ ]:
# Rodar o Optuna
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)


print("Melhores parâmetros encontrados:", study.best_params)
print("Melhor RMSE médio:", study.best_value)

In [ ]:
# Treinar e avaliar no holdout
final_model = xgb.XGBRegressor(**study.best_params)

final_model.fit(X, y)

X_holdout = X_test.values
y_holdout = y_test.values

y_pred_holdout = final_model.predict(X_holdout)
rmse_holdout = root_mean_squared_error(y_holdout, y_pred_holdout)

print(f"\nRMSE no Holdout: {rmse_holdout:.4f}")

### Avaliação em Backtesting do Modelo

In [ ]:
from backtest import Backtester
tester = Backtester(df=test_with_avg, y_pred=y_pred_holdout)
tester.make_backtest()
tester.plot_ev_ts()

In [ ]:
flags = [
    "flag_below_long",
    "flag_above_long",
    "flag_same_result",
    "flag_below_long_1std",
    "flag_above_long_1std",
]

flags_subset = [
    "flag_below_long",
    "flag_above_long",
]


# filtrar quando pelo menos uma dessas flags = 1
df_flags = tester.df[tester.df[flags_subset].any(axis=1)].copy()

# rodar backtest
tester_flag_df = Backtester(df=df_flags, y_pred=df_flags['pred'])
tester_flag_df.make_backtest()
tester_flag_df.plot_ev_ts()

tester_flag_df.resultados


### Salvar objetos do Modelo

In [ ]:
# df_flags.to_excel('df_flags.xlsx', index=False)

In [ ]:
# Salvar Modelo Final
# X_full = np.concatenate([X, X_test], axis=0)
# y_full = np.concatenate([y, y_test], axis=0)

# final_model = xgb.XGBRegressor(**study.best_params)
# final_model.fit(X_full, y_full)
# final_model.save_model("model_22614.json")


# joblib.dump(scaler, "scaler_22614.pkl")

# print("✅ Modelo e scaler salvos com sucesso!")
